# Energy AI Hackathon 2026 - Energy Gladiators

> **SUBMISSION INSTRUCTIONS:**
> 1. Rename this file to `<TeamName>.ipynb` (your exact registered team name)
> 2. Ensure this structure matches `Hackathon_ProjectTemplate.ipynb`
> 3. Fill in all [PLACEHOLDER] sections with actual data
> 4. Commit to the hackathon GitHub organization repo

**Team Members:**
- [Member 1 Full Name] - [Department/Affiliation]
- [Member 2 Full Name] - [Department/Affiliation]
- [Member 3 Full Name] - [Department/Affiliation]
- [Member 4 Full Name] - [Department/Affiliation]

---

## Executive Summary

This notebook presents our complete machine learning workflow for predicting energy usage during hydraulic fracturing operations. Our solution predicts Grid (kWh), Diesel (gal), and CNG (MMBTU) consumption for 50 test wells, including 100 uncertainty realizations per prediction.

**Key Approach:**
- Separate Random Forest models for each fuel type (respecting domain logic)
- Residual bootstrapping for uncertainty quantification
- Feature engineering informed by domain expertise

**Results Summary:**
- [TO BE FILLED WITH 2026 RESULTS]


---
## 1. Problem Statement

The Energy AI Hackathon 2026 challenges teams to predict energy consumption during hydraulic fracturing ("fracking") operations. This is critical for:

- **Cost optimization**: Accurate energy forecasting helps operators plan fuel logistics
- **Environmental planning**: Understanding energy mix impacts emissions calculations
- **Operational efficiency**: Avoiding fuel shortages or costly oversupply

### Prediction Targets
| Target | Unit | Description |
|--------|------|-------------|
| Grid | kWh | Electrical grid energy consumption |
| Diesel | gallons | Diesel fuel consumption |
| CNG | MMBTU | Compressed natural gas consumption |

### Fleet Types and Energy Sources
Each well has a "Fleet Type" that determines which energy sources it uses:

| Fleet Type | Energy Sources | Output Rows |
|------------|----------------|-------------|
| Grid | Electricity only | 1 |
| Diesel | Diesel only | 1 |
| Turbine | CNG only | 1 |
| DGB (Dual-fuel) | Diesel AND CNG | 2 |

This domain knowledge directly informed our modeling strategy: we train separate models per fuel type rather than predicting zeros for unused fuel sources.


---
## 2. Setup and Data Ingestion


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

In [ ]:
train_df = pd.read_csv('data/HackathonData2026.csv')
test_df = pd.read_csv('data/testing2026.csv')

print(f"Training data: {train_df.shape[0]} wells, {train_df.shape[1]} columns")
print(f"Test data: {test_df.shape[0]} wells, {test_df.shape[1]} columns")

In [ ]:
print("\n=== Training Data Overview ===")
display(train_df.head())
print("\n=== Data Types ===")
print(train_df.dtypes)

In [ ]:
print("\n=== Missing Values ===")
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

---
## 3. Data Cleaning and Imputation

We handle missing values using domain-appropriate strategies:
- **Numeric columns**: Median imputation (robust to outliers)
- **Categorical columns**: Mode imputation (most common value)

This ensures no data is lost while maintaining statistical validity.


In [ ]:
def clean_data(df):
    """Apply imputation to handle missing values."""
    df_clean = df.copy()
    
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df_clean[col].isnull().sum() > 0:
            df_clean[col].fillna(df_clean[col].median(), inplace=True)
    
    categorical_cols = df_clean.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        if df_clean[col].isnull().sum() > 0:
            df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)
    
    return df_clean

train_clean = clean_data(train_df)
test_clean = clean_data(test_df)

print(f"Remaining missing values in training: {train_clean.isnull().sum().sum()}")
print(f"Remaining missing values in test: {test_clean.isnull().sum().sum()}")

---
## 4. Exploratory Data Analysis

Understanding the data distribution and relationships before modeling.


In [ ]:
print("=== Target Variable Statistics ===")
target_cols = ['Grid', 'Diesel', 'CNG']
for col in target_cols:
    if col in train_clean.columns:
        non_zero = train_clean[train_clean[col] > 0][col]
        print(f"\n{col}:")
        print(f"  Non-zero count: {len(non_zero)}")
        print(f"  Mean: {non_zero.mean():.2f}")
        print(f"  Std: {non_zero.std():.2f}")
        print(f"  Min: {non_zero.min():.2f}, Max: {non_zero.max():.2f}")

In [ ]:
print("=== Fleet Type Distribution ===")
fleet_col = 'Fleet Type' if 'Fleet Type' in train_clean.columns else 'Fuel Type'
if fleet_col in train_clean.columns:
    print(train_clean[fleet_col].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(['Grid', 'Diesel', 'CNG']):
    if col in train_clean.columns:
        non_zero = train_clean[train_clean[col] > 0][col]
        axes[i].hist(non_zero, bins=30, edgecolor='black', alpha=0.7)
        axes[i].set_title(f'{col} Distribution (non-zero)')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Count')

plt.tight_layout()
plt.savefig('images/energy_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: images/energy_distributions.png")

---
## 5. Feature Engineering

We create domain-informed features that capture operational characteristics:

| Feature | Formula | Rationale |
|---------|---------|----------|
| Time_Overrun | Actual - Estimated Stage Time | Delays may indicate equipment issues, affecting fuel consumption |
| Total_Pumping_Time | Stages × Actual Stage Time | Total operational time correlates with energy usage |
| Clusters_per_Stage | Clusters / Stages | Intensity of fracturing per stage |


In [ ]:
def engineer_features(df):
    """Create domain-informed features."""
    df_feat = df.copy()
    
    stage_col = [c for c in df.columns if 'Stage' in c and '#' in c]
    cluster_col = [c for c in df.columns if 'Cluster' in c and '#' in c]
    est_time_col = [c for c in df.columns if 'Estimated' in c and 'Time' in c]
    act_time_col = [c for c in df.columns if 'Actual' in c and 'Time' in c]
    
    if stage_col:
        stages = stage_col[0]
    else:
        stages = 'Number of Stages' if 'Number of Stages' in df.columns else None
    
    if cluster_col:
        clusters = cluster_col[0]
    else:
        clusters = 'Number of Clusters' if 'Number of Clusters' in df.columns else None
    
    if est_time_col:
        est_time = est_time_col[0]
    else:
        est_time = 'Estimated Avg Stage Time (min)' if 'Estimated Avg Stage Time (min)' in df.columns else None
    
    if act_time_col:
        act_time = act_time_col[0]
    else:
        act_time = 'Actual Avg Stage Time (min)' if 'Actual Avg Stage Time (min)' in df.columns else None
    
    if est_time and act_time:
        df_feat['Time_Overrun'] = df_feat[act_time] - df_feat[est_time]
        print("Created: Time_Overrun")
    
    if stages and act_time:
        df_feat['Total_Pumping_Time'] = df_feat[stages] * df_feat[act_time]
        print("Created: Total_Pumping_Time")
    
    if clusters and stages:
        df_feat['Clusters_per_Stage'] = df_feat[clusters] / df_feat[stages].replace(0, 1)
        print("Created: Clusters_per_Stage")
    
    return df_feat

train_feat = engineer_features(train_clean)
test_feat = engineer_features(test_clean)

---
## 6. Model Training

### Modeling Strategy

We train **separate Random Forest models for each fuel type**. This domain-driven decision:
- Avoids predicting zero values for unused fuel sources
- Allows each model to specialize in its fuel type's patterns
- Handles DGB (dual-fuel) wells by generating both Diesel and CNG predictions

### Why Random Forest?
- Robust to outliers and noisy data
- Handles mixed feature types (numeric + categorical)
- Provides feature importance for interpretability
- Proven effective in previous hackathons


In [ ]:
def prepare_features(df, target_cols=['Grid', 'Diesel', 'CNG']):
    """Prepare feature matrix by encoding categoricals and dropping targets."""
    df_prep = df.copy()
    
    drop_cols = target_cols + ['Well Name', 'Masked Well Name']
    drop_cols = [c for c in drop_cols if c in df_prep.columns]
    
    X = df_prep.drop(columns=drop_cols, errors='ignore')
    
    label_encoders = {}
    for col in X.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        label_encoders[col] = le
    
    return X, label_encoders

X_train, encoders = prepare_features(train_feat)
print(f"Feature matrix shape: {X_train.shape}")
print(f"Features: {list(X_train.columns)}")

In [ ]:
def train_fuel_model(X, y, fuel_name, n_estimators=100, max_depth=15):
    """Train a Random Forest model for a specific fuel type."""
    
    mask = y > 0
    X_fuel = X[mask]
    y_fuel = y[mask]
    
    print(f"\n=== Training {fuel_name} Model ===")
    print(f"Training samples (non-zero): {len(y_fuel)}")
    
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42,
        n_jobs=-1
    )
    
    cv_scores = cross_val_score(model, X_fuel, y_fuel, cv=5, scoring='r2')
    print(f"Cross-validation R² scores: {cv_scores.round(3)}")
    print(f"Mean CV R²: {cv_scores.mean():.3f} (+/- {cv_scores.std()*2:.3f})")
    
    cv_predictions = cross_val_predict(model, X_fuel, y_fuel, cv=5)
    residuals = y_fuel.values - cv_predictions
    
    model.fit(X_fuel, y_fuel)
    
    return model, residuals, cv_scores.mean()

models = {}
residuals = {}
cv_scores = {}

for fuel in ['Grid', 'Diesel', 'CNG']:
    if fuel in train_feat.columns:
        y = train_feat[fuel]
        models[fuel], residuals[fuel], cv_scores[fuel] = train_fuel_model(X_train, y, fuel)

In [ ]:
print("\n" + "="*50)
print("MODEL PERFORMANCE SUMMARY")
print("="*50)
for fuel, score in cv_scores.items():
    print(f"{fuel:10} CV R²: {score:.3f}")

---
## 7. Feature Importance Analysis

Understanding which features drive predictions helps validate our model and provides insights for operations.


In [ ]:
def plot_feature_importance(model, feature_names, fuel_name, top_n=10):
    """Plot top N most important features."""
    importance = pd.DataFrame({
        'Feature': feature_names,
        'Importance': model.feature_importances_
    }).sort_values('Importance', ascending=False).head(top_n)
    
    plt.figure(figsize=(10, 6))
    plt.barh(importance['Feature'], importance['Importance'])
    plt.xlabel('Importance')
    plt.title(f'Top {top_n} Feature Importance - {fuel_name} Model')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(f'images/feature_importance_{fuel_name.lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return importance

import os
os.makedirs('images', exist_ok=True)

for fuel, model in models.items():
    importance = plot_feature_importance(model, X_train.columns, fuel)
    print(f"\nTop features for {fuel}:")
    print(importance.to_string(index=False))

---
## 8. Uncertainty Quantification

### Method: Residual Bootstrapping

We generate 100 uncertainty realizations per prediction using residual bootstrapping:

1. **Compute residuals** from cross-validation predictions
2. **Sample residuals** with replacement (100 times per prediction)
3. **Add to point estimate** to create uncertainty distribution

This method:
- Captures realistic prediction uncertainty based on actual model errors
- Is computationally efficient
- Produces physically meaningful uncertainty ranges


In [ ]:
def generate_predictions_with_uncertainty(model, X, residuals, n_realizations=100):
    """Generate point estimates and uncertainty realizations."""
    
    point_estimates = model.predict(X)
    
    n_samples = len(point_estimates)
    realizations = np.zeros((n_samples, n_realizations))
    
    for i in range(n_samples):
        sampled_residuals = np.random.choice(residuals, size=n_realizations, replace=True)
        realizations[i, :] = point_estimates[i] + sampled_residuals
    
    realizations = np.maximum(realizations, 0)
    
    return point_estimates, realizations

print("Uncertainty quantification method ready.")

---
## 9. Generate Predictions for Test Wells


In [ ]:
X_test, _ = prepare_features(test_feat)

for col in X_train.columns:
    if col not in X_test.columns:
        X_test[col] = 0
X_test = X_test[X_train.columns]

print(f"Test feature matrix shape: {X_test.shape}")

In [ ]:
fleet_col = 'Fleet Type' if 'Fleet Type' in test_feat.columns else 'Fuel Type'
well_col = 'Masked Well Name' if 'Masked Well Name' in test_feat.columns else 'Well Name'

solution_rows = []

for idx, row in test_feat.iterrows():
    well_name = row[well_col]
    fleet_type = row[fleet_col]
    X_well = X_test.loc[[idx]]
    
    if fleet_type == 'Grid':
        point, reals = generate_predictions_with_uncertainty(models['Grid'], X_well, residuals['Grid'])
        solution_rows.append({
            'Masked Well Name': well_name,
            'Fuel Type': 'Grid',
            'Fuel Value': point[0],
            **{f'R_{i+1}': reals[0, i] for i in range(100)}
        })
    
    elif fleet_type == 'Diesel':
        point, reals = generate_predictions_with_uncertainty(models['Diesel'], X_well, residuals['Diesel'])
        solution_rows.append({
            'Masked Well Name': well_name,
            'Fuel Type': 'Diesel',
            'Fuel Value': point[0],
            **{f'R_{i+1}': reals[0, i] for i in range(100)}
        })
    
    elif fleet_type == 'Turbine':
        point, reals = generate_predictions_with_uncertainty(models['CNG'], X_well, residuals['CNG'])
        solution_rows.append({
            'Masked Well Name': well_name,
            'Fuel Type': 'Turbine',
            'Fuel Value': point[0],
            **{f'R_{i+1}': reals[0, i] for i in range(100)}
        })
    
    elif fleet_type == 'DGB':
        point_d, reals_d = generate_predictions_with_uncertainty(models['Diesel'], X_well, residuals['Diesel'])
        solution_rows.append({
            'Masked Well Name': well_name,
            'Fuel Type': 'DGB_Diesel',
            'Fuel Value': point_d[0],
            **{f'R_{i+1}': reals_d[0, i] for i in range(100)}
        })
        
        point_c, reals_c = generate_predictions_with_uncertainty(models['CNG'], X_well, residuals['CNG'])
        solution_rows.append({
            'Masked Well Name': well_name,
            'Fuel Type': 'DGB_CNG',
            'Fuel Value': point_c[0],
            **{f'R_{i+1}': reals_c[0, i] for i in range(100)}
        })

solution_df = pd.DataFrame(solution_rows)
print(f"Solution shape: {solution_df.shape}")
print(f"Expected: 63 rows for 50 wells (DGB wells have 2 rows)")

In [ ]:
solution_df.to_csv('outputs/solution.csv', index=False)
print("Saved: outputs/solution.csv")
print("\nSolution preview:")
display(solution_df[['Masked Well Name', 'Fuel Type', 'Fuel Value', 'R_1', 'R_2', 'R_100']].head(10))

---
## 10. Results Visualization


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fuel_types = solution_df['Fuel Type'].value_counts()
axes[0].bar(fuel_types.index, fuel_types.values)
axes[0].set_title('Predictions by Fuel Type')
axes[0].set_xlabel('Fuel Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

for fuel_type in solution_df['Fuel Type'].unique():
    subset = solution_df[solution_df['Fuel Type'] == fuel_type]['Fuel Value']
    axes[1].hist(subset, alpha=0.5, label=fuel_type, bins=15)
axes[1].set_title('Distribution of Point Estimates')
axes[1].set_xlabel('Fuel Value')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('images/prediction_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
sample_wells = solution_df.head(5)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for i, (idx, row) in enumerate(sample_wells.iterrows()):
    reals = [row[f'R_{j}'] for j in range(1, 101)]
    axes[i].hist(reals, bins=20, edgecolor='black', alpha=0.7)
    axes[i].axvline(row['Fuel Value'], color='red', linestyle='--', linewidth=2, label='Point Est.')
    axes[i].set_title(f"{row['Fuel Type']}\n{row['Masked Well Name'][:15]}...")
    axes[i].set_xlabel('Value')
    if i == 0:
        axes[i].set_ylabel('Count')

plt.suptitle('Uncertainty Distributions for Sample Wells (100 Realizations)', fontsize=12)
plt.tight_layout()
plt.savefig('images/uncertainty_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. Conclusions

### Summary

We developed a complete machine learning workflow for predicting energy usage during hydraulic fracturing operations:

**Model Performance:**
- [TO BE FILLED: Grid R², Diesel R², CNG R²]

**Key Findings:**
- [TO BE FILLED: Top predictive features]
- [TO BE FILLED: Insights from feature importance]

**Methodology Strengths:**
- Separate models per fuel type respect domain logic
- Residual bootstrapping provides realistic uncertainty quantification
- Feature engineering captures operational characteristics

### Real-World Impact

Accurate energy predictions with uncertainty bounds enable operators to:
- **Plan fuel logistics** with confidence intervals
- **Avoid costly shortages** by understanding prediction uncertainty
- **Optimize costs** by not over-ordering fuel based on worst-case estimates

---
*Notebook created for Energy AI Hackathon 2026*
